# YMIR Feature Extraction with Stratified 5-Fold Cross-Validation

This notebook creates leakage-free cross-validation folds at the original 30-second recording level. Segmentation is performed only after fold assignment.

In [1]:

# ============================================================
# YMIR Feature Extraction with Stratified 5-Fold Cross-Validation
# Leakage-free protocol:
#   original 30-second files -> StratifiedKFold -> segmentation inside each fold
# ============================================================

import os
import csv
import gc
import numpy as np
import librosa
import tensorflow as tf
from sklearn.model_selection import StratifiedKFold

# ============================================================
# Settings
# ============================================================

SR = 22050
FILE_DURATION = 30.0
SEGMENTS = 5
SEGMENT_DURATION = FILE_DURATION / SEGMENTS
IMG_SIZE = (224, 224)

N_SPLITS = 5
RANDOM_STATE = 42

DATA_PATH = "/root/YMED/"
GENRES = ["adeni", "hadrami", "lahji", "sanaai", "tihami"]

SAVE_DIR = "/root/autodl-tmp/cv_split_data/"
os.makedirs(SAVE_DIR, exist_ok=True)

AVAILABLE_FEATURES = [
    "mfcc13",
    "mfcc20",
    "mfcc40",
    "chroma",
    "mel_spectrogram",
    "filterbank"
]

# ============================================================
# Audio loading and segmentation
# ============================================================

def load_audio_30s(file_path, sr=SR, duration=FILE_DURATION):
    """
    Load audio at a fixed sampling rate and force it to exactly 30 seconds.
    Short files are zero-padded; long files are truncated.
    """
    target_len = int(sr * duration)
    y_audio, _ = librosa.load(file_path, sr=sr, mono=True)

    if len(y_audio) < target_len:
        y_audio = np.pad(y_audio, (0, target_len - len(y_audio)), mode="constant")
    else:
        y_audio = y_audio[:target_len]

    return y_audio.astype(np.float32)


def extract_segments(y_audio, sr=SR):
    """
    Split one 30-second audio signal into five non-overlapping 6-second segments.
    """
    segment_len = int(SEGMENT_DURATION * sr)
    segments = []

    for i in range(SEGMENTS):
        start = i * segment_len
        end = start + segment_len
        segment = y_audio[start:end]

        if len(segment) == segment_len:
            segments.append(segment.astype(np.float32))

    return segments


# ============================================================
# Feature extraction
# ============================================================

def extract_feature(seg, feature_type="mel_spectrogram", sr=SR):
    """
    Extract one feature representation from one 6-second segment.
    """
    if feature_type == "mfcc13":
        feature = librosa.feature.mfcc(y=seg, sr=sr, n_mfcc=13)

    elif feature_type == "mfcc20":
        feature = librosa.feature.mfcc(y=seg, sr=sr, n_mfcc=20)

    elif feature_type == "mfcc40":
        feature = librosa.feature.mfcc(y=seg, sr=sr, n_mfcc=40)

    elif feature_type == "chroma":
        feature = librosa.feature.chroma_stft(y=seg, sr=sr)

    elif feature_type == "mel_spectrogram":
        mel = librosa.feature.melspectrogram(y=seg, sr=sr, n_mels=128)
        feature = librosa.power_to_db(mel, ref=np.max)

    elif feature_type == "filterbank":
        mel = librosa.feature.melspectrogram(y=seg, sr=sr, n_mels=40)
        feature = librosa.power_to_db(mel, ref=np.max)

    else:
        raise ValueError(f"Unknown feature type: {feature_type}")

    return feature.astype(np.float32)


def prepare_feature_image(feature):
    """
    Resize feature map to 224x224 and convert it to 3 channels.
    """
    feature = tf.image.resize(feature[..., np.newaxis], IMG_SIZE).numpy()
    feature = np.repeat(feature, 3, axis=-1)
    return feature.astype(np.float32)


# ============================================================
# Optional augmentation: training folds only
# ============================================================

def fix_length_audio(y, target_len):
    if len(y) > target_len:
        y = y[:target_len]
    elif len(y) < target_len:
        y = np.pad(y, (0, target_len - len(y)), mode="constant")
    return y.astype(np.float32)


def augment_waveform_segment(seg, sr=SR, seed=None):
    """
    Return original + waveform-level augmentations.
    Applied only to training folds when enabled.
    """
    rng = np.random.default_rng(seed)
    target_len = len(seg)
    augmented_segments = []

    augmented_segments.append(("original", seg.astype(np.float32)))

    # Time shift
    try:
        max_shift = int(0.10 * target_len)
        shift = int(rng.integers(-max_shift, max_shift + 1))
        augmented_segments.append(("time_shift", np.roll(seg, shift).astype(np.float32)))
    except Exception as e:
        print(f"Time shift failed: {e}")

    # Pitch shift
    try:
        n_steps = float(rng.choice([-2, -1, 1, 2]))
        pitched = librosa.effects.pitch_shift(y=seg, sr=sr, n_steps=n_steps)
        augmented_segments.append(("pitch_shift", fix_length_audio(pitched, target_len)))
    except Exception as e:
        print(f"Pitch shift failed: {e}")

    # Time stretch
    try:
        rate = float(rng.uniform(0.90, 1.10))
        stretched = librosa.effects.time_stretch(y=seg, rate=rate)
        augmented_segments.append(("time_stretch", fix_length_audio(stretched, target_len)))
    except Exception as e:
        print(f"Time stretch failed: {e}")

    # Noise injection
    try:
        noise_level = float(rng.uniform(0.003, 0.008))
        noise = noise_level * rng.standard_normal(target_len)
        augmented_segments.append(("noise_injection", fix_length_audio(seg + noise, target_len)))
    except Exception as e:
        print(f"Noise injection failed: {e}")

    return augmented_segments


def spec_augment(feature, max_freq_mask=12, max_time_mask=20, seed=None):
    """
    Apply simple SpecAugment to a 2D feature map.
    Applied only to training folds when enabled.
    """
    rng = np.random.default_rng(seed)
    feature = feature.copy()

    num_freq, num_time = feature.shape

    f = int(rng.integers(0, min(max_freq_mask, num_freq) + 1))
    if f > 0:
        f0 = int(rng.integers(0, max(1, num_freq - f)))
        feature[f0:f0 + f, :] = 0

    t = int(rng.integers(0, min(max_time_mask, num_time) + 1))
    if t > 0:
        t0 = int(rng.integers(0, max(1, num_time - t)))
        feature[:, t0:t0 + t] = 0

    return feature.astype(np.float32)


# ============================================================
# File collection and leakage check
# ============================================================

def collect_original_files(data_path, genres):
    """
    Collect original 30-second files before segmentation.
    """
    file_paths = []
    labels = []
    genre_names = []

    for idx, genre in enumerate(genres):
        genre_path = os.path.join(data_path, genre)

        if not os.path.isdir(genre_path):
            raise FileNotFoundError(f"Genre folder not found: {genre_path}")

        wav_files = sorted([
            f for f in os.listdir(genre_path)
            if f.lower().endswith(".wav")
        ])

        print(f"{genre}: {len(wav_files)} original files")

        for filename in wav_files:
            file_paths.append(os.path.join(genre_path, filename))
            labels.append(idx)
            genre_names.append(genre)

    return np.array(file_paths), np.array(labels), np.array(genre_names)


def check_no_data_leakage(train_files, test_files):
    train_set = set(os.path.basename(f) for f in train_files)
    test_set = set(os.path.basename(f) for f in test_files)
    overlap = train_set.intersection(test_set)

    if overlap:
        raise RuntimeError(f"Data leakage detected: {list(overlap)[:10]}")

    print("Data leakage check: PASSED")


def save_metadata(metadata, save_path):
    if not metadata:
        return

    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    keys = metadata[0].keys()

    with open(save_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=keys)
        writer.writeheader()
        writer.writerows(metadata)


# ============================================================
# Fold processing
# ============================================================

def process_files_to_arrays(
    file_paths,
    labels,
    feature_type,
    split_name,
    fold_id,
    augment_train=False,
    apply_specaugment=False
):
    """
    Segment original files and extract feature images.
    Segmentation occurs only after fold assignment.
    """
    X = []
    y = []
    metadata = []

    for file_idx, (file_path, label) in enumerate(zip(file_paths, labels)):
        print(f"[Fold {fold_id} | {split_name} | {feature_type}] "
              f"{file_idx + 1}/{len(file_paths)}: {os.path.basename(file_path)}")

        try:
            y_audio = load_audio_30s(file_path)
            segments = extract_segments(y_audio)

            for seg_id, seg in enumerate(segments):
                if split_name == "train" and augment_train:
                    aug_segments = augment_waveform_segment(
                        seg,
                        seed=RANDOM_STATE + fold_id * 1000 + file_idx * 10 + seg_id
                    )
                else:
                    aug_segments = [("original", seg)]

                for aug_name, aug_seg in aug_segments:
                    feature = extract_feature(aug_seg, feature_type=feature_type)

                    if split_name == "train" and apply_specaugment:
                        feature = spec_augment(
                            feature,
                            seed=RANDOM_STATE + fold_id * 10000 + file_idx * 100 + seg_id
                        )

                    feature_img = prepare_feature_image(feature)

                    X.append(feature_img)
                    y.append(int(label))

                    metadata.append({
                        "fold": fold_id,
                        "split": split_name,
                        "original_file": os.path.basename(file_path),
                        "file_path": file_path,
                        "segment_id": seg_id,
                        "augmentation": aug_name,
                        "label": int(label),
                        "genre": GENRES[int(label)]
                    })

        except Exception as e:
            print(f"Skipping file due to error: {file_path}")
            print(f"Error: {e}")

    return np.array(X, dtype=np.float32), np.array(y, dtype=np.int64), metadata


def save_fold_arrays(
    X_train,
    y_train,
    X_test,
    y_test,
    train_metadata,
    test_metadata,
    fold_id,
    feature_type,
    save_dir=SAVE_DIR
):
    fold_dir = os.path.join(save_dir, f"fold_{fold_id}")
    train_dir = os.path.join(fold_dir, "train", feature_type)
    test_dir = os.path.join(fold_dir, "test", feature_type)
    meta_dir = os.path.join(fold_dir, "metadata", feature_type)

    os.makedirs(train_dir, exist_ok=True)
    os.makedirs(test_dir, exist_ok=True)
    os.makedirs(meta_dir, exist_ok=True)

    np.save(os.path.join(train_dir, f"{feature_type}_X_train.npy"), X_train)
    np.save(os.path.join(train_dir, f"{feature_type}_y_train.npy"), y_train)
    np.save(os.path.join(test_dir, f"{feature_type}_X_test.npy"), X_test)
    np.save(os.path.join(test_dir, f"{feature_type}_y_test.npy"), y_test)

    save_metadata(train_metadata, os.path.join(meta_dir, f"{feature_type}_train_metadata.csv"))
    save_metadata(test_metadata, os.path.join(meta_dir, f"{feature_type}_test_metadata.csv"))

    print(f"\nSaved Fold {fold_id} | {feature_type}")
    print("Training Data Shape:", X_train.shape)
    print("Testing Data Shape :", X_test.shape)


# ============================================================
# Main cross-validation feature extraction
# ============================================================

def run_cv_feature_extraction(
    data_path=DATA_PATH,
    genres=GENRES,
    features=None,
    n_splits=N_SPLITS,
    save_dir=SAVE_DIR,
    augment_train=False,
    apply_specaugment=False
):
    """
    Create 5-fold cross-validation feature arrays.
    The folds are created at the original recording level.
    """
    if features is None:
        features = ["mel_spectrogram"]

    print("=" * 80)
    print("YMIR Feature Extraction with Stratified 5-Fold Cross-Validation")
    print("=" * 80)

    file_paths, labels, genre_names = collect_original_files(data_path, genres)

    print("\nTotal original files:", len(file_paths))

    skf = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=RANDOM_STATE
    )

    split_records = []

    for fold_id, (train_idx, test_idx) in enumerate(skf.split(file_paths, labels), start=1):
        print("\n" + "=" * 80)
        print(f"Fold {fold_id}/{n_splits}")
        print("=" * 80)

        train_files = file_paths[train_idx]
        test_files = file_paths[test_idx]
        train_labels = labels[train_idx]
        test_labels = labels[test_idx]

        check_no_data_leakage(train_files, test_files)

        print("Train original files:", len(train_files))
        print("Test original files :", len(test_files))

        for fp, lb in zip(train_files, train_labels):
            split_records.append({
                "fold": fold_id,
                "split": "train",
                "file_path": fp,
                "file_name": os.path.basename(fp),
                "label": int(lb),
                "genre": genres[int(lb)]
            })

        for fp, lb in zip(test_files, test_labels):
            split_records.append({
                "fold": fold_id,
                "split": "test",
                "file_path": fp,
                "file_name": os.path.basename(fp),
                "label": int(lb),
                "genre": genres[int(lb)]
            })

        for feature_type in features:
            print("\n" + "-" * 70)
            print(f"Processing Fold {fold_id} | Feature: {feature_type}")
            print("-" * 70)

            X_train, y_train, train_metadata = process_files_to_arrays(
                train_files,
                train_labels,
                feature_type=feature_type,
                split_name="train",
                fold_id=fold_id,
                augment_train=augment_train,
                apply_specaugment=apply_specaugment
            )

            X_test, y_test, test_metadata = process_files_to_arrays(
                test_files,
                test_labels,
                feature_type=feature_type,
                split_name="test",
                fold_id=fold_id,
                augment_train=False,
                apply_specaugment=False
            )

            save_fold_arrays(
                X_train,
                y_train,
                X_test,
                y_test,
                train_metadata,
                test_metadata,
                fold_id,
                feature_type,
                save_dir=save_dir
            )

            del X_train, y_train, X_test, y_test
            gc.collect()

    split_csv = os.path.join(save_dir, "cv_original_file_splits.csv")
    save_metadata(split_records, split_csv)
    print("\nOriginal-file CV split metadata saved to:", split_csv)
    print("\nFeature extraction completed.")


# ============================================================
# Execute
# ============================================================

if __name__ == "__main__":
    print("\nAvailable Features:")
    for i, feat in enumerate(AVAILABLE_FEATURES, 1):
        print(f"  {i}. {feat}")

    choice = input("\nSelect features for CV extraction (comma-separated numbers or 'all', default=5): ").strip()

    if choice.lower() == "all":
        selected_features = AVAILABLE_FEATURES
    elif choice == "":
        selected_features = ["mel_spectrogram"]
    else:
        indices = [int(x.strip()) - 1 for x in choice.split(",")]
        selected_features = [AVAILABLE_FEATURES[i] for i in indices if 0 <= i < len(AVAILABLE_FEATURES)]

    aug_choice = input("\nApply waveform augmentation to training folds? (yes/no, default=no): ").strip().lower() or "no"
    augment_train = aug_choice == "yes"

    spec_choice = input("Apply SpecAugment to training feature maps? (yes/no, default=no): ").strip().lower() or "no"
    apply_specaugment = spec_choice == "yes"

    print("\nSelected features:", selected_features)
    print("Waveform augmentation:", augment_train)
    print("SpecAugment:", apply_specaugment)
    print("Output directory:", SAVE_DIR)

    run_cv_feature_extraction(
        features=selected_features,
        augment_train=augment_train,
        apply_specaugment=apply_specaugment
    )



Available Features:
  1. mfcc13
  2. mfcc20
  3. mfcc40
  4. chroma
  5. mel_spectrogram
  6. filterbank



Select features for CV extraction (comma-separated numbers or 'all', default=5):  1

Apply waveform augmentation to training folds? (yes/no, default=no):  
Apply SpecAugment to training feature maps? (yes/no, default=no):  



Selected features: ['mfcc13']
Waveform augmentation: False
SpecAugment: False
Output directory: /root/autodl-tmp/cv_split_data/
YMIR Feature Extraction with Stratified 5-Fold Cross-Validation
adeni: 230 original files
hadrami: 230 original files
lahji: 230 original files
sanaai: 230 original files
tihami: 230 original files

Total original files: 1150

Fold 1/5
Data leakage check: PASSED
Train original files: 920
Test original files : 230

----------------------------------------------------------------------
Processing Fold 1 | Feature: mfcc13
----------------------------------------------------------------------
[Fold 1 | train | mfcc13] 1/920: 0044_0001_177_54_3.wav


2026-05-23 03:55:03.723233: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-05-23 03:55:04.239419: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1532] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 30416 MB memory:  -> device: 0, name: NVIDIA vGPU-32GB, pci bus id: 0000:c1:00.0, compute capability: 8.9


[Fold 1 | train | mfcc13] 2/920: 0045_0002_177_54_3.wav
[Fold 1 | train | mfcc13] 3/920: 0047_0004_177_54_3.wav
[Fold 1 | train | mfcc13] 4/920: 0049_0006_177_54_3.wav
[Fold 1 | train | mfcc13] 5/920: 0051_0008_177_54_3.wav
[Fold 1 | train | mfcc13] 6/920: 0052_0009_177_54_3.wav
[Fold 1 | train | mfcc13] 7/920: 0053_0010_177_54_3.wav
[Fold 1 | train | mfcc13] 8/920: 0054_0011_177_54_3.wav
[Fold 1 | train | mfcc13] 9/920: 0056_0013_177_54_3.wav
[Fold 1 | train | mfcc13] 10/920: 0057_0014_177_54_3.wav
[Fold 1 | train | mfcc13] 11/920: 0058_0015_177_54_3.wav
[Fold 1 | train | mfcc13] 12/920: 0059_0016_177_54_3.wav
[Fold 1 | train | mfcc13] 13/920: 0060_0017_177_54_3.wav
[Fold 1 | train | mfcc13] 14/920: 0070_0001_179_55_3.wav
[Fold 1 | train | mfcc13] 15/920: 0071_0002_179_55_3.wav
[Fold 1 | train | mfcc13] 16/920: 0072_0003_179_55_3.wav
[Fold 1 | train | mfcc13] 17/920: 0073_0004_179_55_3.wav
[Fold 1 | train | mfcc13] 18/920: 0074_0005_179_55_3.wav
[Fold 1 | train | mfcc13] 19/920: 0075_